In [7]:
import argparse
import copy
import math
import os
import random
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

try:
    import torch_pruning as tp
except Exception:
    tp = None

try:
    from thop import profile
except Exception:
    profile = None


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def replace_classifier(model: nn.Module, arch: str, num_classes: int) -> nn.Module:
    arch = arch.lower()
    if arch in {"resnet18", "resnet34"}:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch == "densenet121":
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    else:
        raise ValueError(f"Unsupported architecture: {arch}")
    return model


def build_model(arch: str, num_classes: int) -> nn.Module:
    arch = arch.lower()
    if arch == "resnet18":
        model = models.resnet18(pretrained=None)
    elif arch == "resnet34":
        model = models.resnet34(pretrained=None)
    elif arch == "densenet121":
        model = models.densenet121(pretrained=None)
    else:
        raise ValueError(f"Unsupported architecture: {arch}")
    return replace_classifier(model, arch, num_classes)


def get_cifar_loaders(
    data_root: str,
    dataset_name: str,
    batch_size: int,
    num_workers: int,
    pin_memory: bool,
) -> Tuple[DataLoader, DataLoader, int]:
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    train_tf = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )
    test_tf = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    )

    name = dataset_name.lower()
    if name == "cifar10":
        train_ds = datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tf)
        test_ds = datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tf)
        num_classes = 10
    elif name == "cifar100":
        train_ds = datasets.CIFAR100(root=data_root, train=True, download=True, transform=train_tf)
        test_ds = datasets.CIFAR100(root=data_root, train=False, download=True, transform=test_tf)
        num_classes = 100
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )
    return train_loader, test_loader, num_classes


def topk_correct(outputs: torch.Tensor, targets: torch.Tensor, k: int = 5) -> int:
    _, pred = outputs.topk(k, dim=1)
    return pred.eq(targets.view(-1, 1)).sum().item()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, float]:
    model.eval()
    total = 0
    total_loss = 0.0
    top1 = 0
    top5 = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        total += y.size(0)
        total_loss += loss.item() * y.size(0)
        top1 += out.argmax(dim=1).eq(y).sum().item()
        top5 += topk_correct(out, y, k=5)
    return {
        "loss": total_loss / max(total, 1),
        "top1": 100.0 * top1 / max(total, 1),
        "top5": 100.0 * top5 / max(total, 1),
    }


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


class CompactKernelConv2d(nn.Module):
    """
    Compact convolution using per-connection kernel bank references.
    - out_idx[k], in_idx[k]: connection (output channel, input channel)
    - bank_idx[k]: which kernel in kernel_bank this connection uses
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: Tuple[int, int],
        stride: Tuple[int, int],
        padding: Tuple[int, int],
        dilation: Tuple[int, int],
        bias: Optional[torch.Tensor],
        out_idx: torch.Tensor,
        in_idx: torch.Tensor,
        bank_idx: torch.Tensor,
        kernel_bank: torch.Tensor,
    ) -> None:
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.kernel_size = (int(kernel_size[0]), int(kernel_size[1]))
        self.stride = (int(stride[0]), int(stride[1]))
        self.padding = (int(padding[0]), int(padding[1]))
        self.dilation = (int(dilation[0]), int(dilation[1]))
        self.kernel_elements = self.kernel_size[0] * self.kernel_size[1]

        self.register_buffer("out_idx", out_idx.long())
        self.register_buffer("in_idx", in_idx.long())
        self.register_buffer("bank_idx", bank_idx.long())
        self.kernel_bank = nn.Parameter(kernel_bank)
        if bias is None:
            self.bias = None
        else:
            self.bias = nn.Parameter(bias.clone())

    @property
    def num_active_kernels(self) -> int:
        return int(self.out_idx.numel())

    @property
    def num_bank_kernels(self) -> int:
        return int(self.kernel_bank.size(0))

    @staticmethod
    def from_conv_representative(conv: nn.Conv2d, keep_idx: torch.Tensor) -> "CompactKernelConv2d":
        cout, cin, kh, kw = conv.weight.shape
        keep_idx = keep_idx.long().to(conv.weight.device)
        out_idx = torch.div(keep_idx, cin, rounding_mode="trunc")
        in_idx = keep_idx % cin
        bank_idx = torch.arange(keep_idx.numel(), device=conv.weight.device)
        kernel_bank = conv.weight.data.view(cout * cin, kh, kw)[keep_idx].clone()
        bias = conv.bias.data if conv.bias is not None else None
        return CompactKernelConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=(kh, kw),
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            bias=bias,
            out_idx=out_idx,
            in_idx=in_idx,
            bank_idx=bank_idx,
            kernel_bank=kernel_bank,
        )

    @staticmethod
    def from_conv_centroid(conv: nn.Conv2d, groups_global: List[List[int]]) -> "CompactKernelConv2d":
        cout, cin, kh, kw = conv.weight.shape
        total = cout * cin
        w_flat = conv.weight.data.view(total, kh, kw)
        device = conv.weight.device

        group_members = set()
        bank_kernels: List[torch.Tensor] = []
        bank_assign = torch.full((total,), -1, dtype=torch.long, device=device)

        # One shared centroid per group.
        for g in groups_global:
            ids = torch.tensor(g, device=device, dtype=torch.long)
            centroid = w_flat[ids].mean(dim=0)
            bank_id = len(bank_kernels)
            bank_kernels.append(centroid)
            bank_assign[ids] = bank_id
            for idx in g:
                group_members.add(int(idx))

        # Non-group kernels keep their own unique bank entry.
        for idx in range(total):
            if idx in group_members:
                continue
            bank_id = len(bank_kernels)
            bank_kernels.append(w_flat[idx].clone())
            bank_assign[idx] = bank_id

        active_idx = torch.arange(total, device=device, dtype=torch.long)
        out_idx = torch.div(active_idx, cin, rounding_mode="trunc")
        in_idx = active_idx % cin
        bank_idx = bank_assign[active_idx]
        kernel_bank = torch.stack(bank_kernels, dim=0) if bank_kernels else w_flat.new_zeros((0, kh, kw))
        bias = conv.bias.data if conv.bias is not None else None

        return CompactKernelConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=(kh, kw),
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            bias=bias,
            out_idx=out_idx,
            in_idx=in_idx,
            bank_idx=bank_idx,
            kernel_bank=kernel_bank,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        kh, kw = self.kernel_size
        full_w = x.new_zeros((self.out_channels, self.in_channels, kh, kw))
        if self.num_active_kernels > 0:
            full_w[self.out_idx, self.in_idx, :, :] = self.kernel_bank[self.bank_idx]
        return F.conv2d(
            x,
            full_w,
            self.bias,
            stride=self.stride,
            padding=self.padding,
            dilation=self.dilation,
        )


@torch.no_grad()
def compute_flops(model: nn.Module, device: torch.device) -> float:
    dummy = torch.randn(1, 3, 224, 224, device=device)

    has_compact_conv = any(isinstance(m, CompactKernelConv2d) for m in model.modules())

    # THOP may mis-handle custom compact convs; force analytic fallback in that case.
    if profile is not None and not has_compact_conv:
        try:
            flops, _ = profile(model, inputs=(dummy,), verbose=False)
            return float(flops)
        except Exception:
            pass

    total = {"flops": 0.0}
    hooks = []

    def conv_hook(module: nn.Conv2d, inputs: Tuple[torch.Tensor], output: torch.Tensor) -> None:
        x = inputs[0]
        n = x.size(0)
        out_h, out_w = output.size(2), output.size(3)
        k_h, k_w = module.kernel_size
        in_per_group = module.in_channels // module.groups
        total["flops"] += float(n * module.out_channels * out_h * out_w * in_per_group * k_h * k_w)

    def compact_hook(module: CompactKernelConv2d, inputs: Tuple[torch.Tensor], output: torch.Tensor) -> None:
        x = inputs[0]
        n = x.size(0)
        out_h, out_w = output.size(2), output.size(3)
        total["flops"] += float(n * out_h * out_w * module.num_active_kernels * module.kernel_elements)

    def linear_hook(module: nn.Linear, inputs: Tuple[torch.Tensor], output: torch.Tensor) -> None:
        x = inputs[0]
        n = x.size(0)
        total["flops"] += float(n * module.in_features * module.out_features)

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, CompactKernelConv2d):
            hooks.append(m.register_forward_hook(compact_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))

    was_training = model.training
    model.eval()
    _ = model(dummy)
    if was_training:
        model.train()
    for h in hooks:
        h.remove()
    return float(total["flops"])


def replace_module_by_name(model: nn.Module, module_name: str, new_module: nn.Module) -> None:
    parent_name, _, child_name = module_name.rpartition(".")
    parent = model.get_submodule(parent_name) if parent_name else model
    if child_name.isdigit() and isinstance(parent, (nn.Sequential, nn.ModuleList)):
        parent[int(child_name)] = new_module
    else:
        setattr(parent, child_name, new_module)


def apply_kernel_prune_mask_from_map(
    model: nn.Module,
    kernel_prune_idx_map: Dict[str, List[int]],
) -> None:
    """
    Apply exact kernel-level removals (as zeroed kernels) from Stage A onto a dense Conv2d model.
    This allows Stage B channel pruning to run on the post-kernel weight state.
    """
    for name, prune_idx in kernel_prune_idx_map.items():
        if not prune_idx:
            continue
        named = dict(model.named_modules())
        if name not in named or not isinstance(named[name], nn.Conv2d):
            continue
        conv = named[name]
        cin = int(conv.in_channels)
        with torch.no_grad():
            for idx in prune_idx:
                out_c = int(idx // cin)
                in_c = int(idx % cin)
                conv.weight.data[out_c, in_c] = 0.0


def get_conv_layer_names(model: nn.Module) -> List[str]:
    return [name for name, m in model.named_modules() if isinstance(m, nn.Conv2d)]


def _connected_groups(sim: torch.Tensor, threshold: float) -> List[List[int]]:
    n = sim.size(0)
    if n < 2:
        return []
    parent = list(range(n))

    def find(a: int) -> int:
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    edges = torch.nonzero(torch.triu(sim > threshold, diagonal=1), as_tuple=False).tolist()
    for i, j in edges:
        union(i, j)

    groups: Dict[int, List[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return [g for g in groups.values() if len(g) > 1]


def find_redundant_kernels(
    conv: nn.Conv2d,
    tau_s: float,
    norm_percent: float,
    max_candidates: int,
    min_keep: int,
    max_kernel_prune_frac_per_layer: float = 1.0,
) -> Dict[str, object]:
    w = conv.weight.data
    cout, cin, _, _ = w.shape
    kernels = w.view(cout * cin, -1)
    total_kernels = int(cout * cin)
    if total_kernels <= min_keep:
        return {
            "prune_idx": [],
            "groups_global": [],
            "num_groups": 0,
            "num_candidates": 0,
            "grouped_kernels": 0,
            "kernels_before": total_kernels,
            "out_channels": int(cout),
            "in_channels": int(cin),
        }

    norms = torch.norm(kernels, p=2, dim=1)
    tau_n = torch.quantile(norms, norm_percent / 100.0)
    cand_idx = torch.where(norms <= tau_n)[0]

    if cand_idx.numel() > max_candidates:
        keep_local = torch.argsort(norms[cand_idx])[:max_candidates]
        cand_idx = cand_idx[keep_local]

    if cand_idx.numel() < 2:
        return {
            "prune_idx": [],
            "groups_global": [],
            "num_groups": 0,
            "num_candidates": int(cand_idx.numel()),
            "grouped_kernels": 0,
            "kernels_before": total_kernels,
            "out_channels": int(cout),
            "in_channels": int(cin),
        }

    x = kernels[cand_idx]
    x = x / (torch.norm(x, p=2, dim=1, keepdim=True) + 1e-12)
    sim = x @ x.T
    groups_local = _connected_groups(sim, tau_s)

    groups_global: List[List[int]] = []
    prune_idx: List[int] = []
    grouped = 0
    for g in groups_local:
        ids = cand_idx[torch.tensor(g, device=cand_idx.device)]
        ids_list = [int(z.item()) for z in ids]
        groups_global.append(ids_list)
        local_norms = norms[ids]
        keep = ids[torch.argmax(local_norms)]
        for idx in ids:
            if int(idx.item()) != int(keep.item()):
                prune_idx.append(int(idx.item()))
        grouped += len(g)

    prune_idx = sorted(set(prune_idx))
    max_prunable = max(0, total_kernels - min_keep)

    def _trim_prune_to_count(idxs: List[int], cap: int) -> List[int]:
        if cap < 0 or len(idxs) <= cap:
            return idxs
        pt = torch.tensor(idxs, device=norms.device, dtype=torch.long)
        pn = norms[pt]
        take = torch.argsort(pn)[:cap]
        return sorted(pt[take].tolist())

    if len(prune_idx) > max_prunable:
        prune_idx = _trim_prune_to_count(prune_idx, max_prunable)
        groups_global = []

    if max_kernel_prune_frac_per_layer < 1.0:
        max_by_frac = int(math.floor(max_kernel_prune_frac_per_layer * total_kernels))
        max_by_frac = max(0, min(max_prunable, max_by_frac))
        if len(prune_idx) > max_by_frac:
            prune_idx = _trim_prune_to_count(prune_idx, max_by_frac)
            groups_global = []

    grouped_out = int(grouped) if groups_global else len(prune_idx)
    return {
        "prune_idx": prune_idx,
        "groups_global": groups_global,
        "num_groups": len(groups_global),
        "num_candidates": int(cand_idx.numel()),
        "grouped_kernels": grouped_out,
        "kernels_before": total_kernels,
        "out_channels": int(cout),
        "in_channels": int(cin),
    }


def kernel_prune_model(
    model: nn.Module,
    tau_s: float = 0.90,
    norm_percent: float = 40.0,
    max_candidates_per_layer: int = 1000,
    min_keep_channels: int = 8,
    kernel_option: str = "representative",
    max_kernel_prune_frac_per_layer: float = 1.0,
    skip_stem: bool = True,
    skip_downsample: bool = True,
) -> Dict[str, object]:
    option = kernel_option.lower()
    if option not in {"representative", "centroid"}:
        raise ValueError("kernel_option must be 'representative' or 'centroid'")

    layer_stats: List[Dict[str, object]] = []
    total_before = 0
    total_removed_effective = 0
    total_active_after = 0
    total_unique_after = 0
    channel_scores: Dict[str, torch.Tensor] = {}
    kernel_prune_idx_map: Dict[str, List[int]] = {}

    conv_names = get_conv_layer_names(model)
    for li, name in enumerate(conv_names):
        named = dict(model.named_modules())
        if name not in named:
            continue
        conv = named[name]
        if not isinstance(conv, nn.Conv2d):
            continue
        if skip_stem and name in {"conv1", "features.conv0"}:
            continue
        if skip_downsample and "downsample" in name:
            continue

        print(f"[Kernel Pruning] Layer {li+1}/{len(conv_names)}: {name}", flush=True)
        info = find_redundant_kernels(
            conv=conv,
            tau_s=tau_s,
            norm_percent=norm_percent,
            max_candidates=max_candidates_per_layer,
            min_keep=min_keep_channels,
            max_kernel_prune_frac_per_layer=max_kernel_prune_frac_per_layer,
        )

        cin = int(conv.in_channels)
        out_pruned = torch.zeros(conv.out_channels, device=conv.weight.device)
        if len(info["prune_idx"]) > 0:
            for idx in info["prune_idx"]:
                out_c = int(idx // cin)
                out_pruned[out_c] += 1.0
        channel_scores[name] = (out_pruned / max(cin, 1)).detach().cpu()
        kernel_prune_idx_map[name] = [int(x) for x in info["prune_idx"]]

        kernels_before = int(info["kernels_before"])
        pruned_count = len(info["prune_idx"])
        groups_count = int(info["num_groups"])
        if info["groups_global"]:
            merged_count = sum(max(0, len(g) - 1) for g in info["groups_global"])
        else:
            merged_count = pruned_count
        if groups_count == 1 and int(info["num_candidates"]) >= 100:
            print(
                "[Kernel Pruning] Warning: one similarity component covers most candidates — "
                "tau_s is likely too low (graph collapses). Per-layer kernel merge cap applies.",
                flush=True,
            )
        print(
            f"Layer: {name} | candidates={info['num_candidates']} | groups={groups_count} | "
            f"effective_removed={merged_count}",
            flush=True,
        )

        use_centroid = option == "centroid" and len(info["groups_global"]) > 0
        if use_centroid:
            compact = CompactKernelConv2d.from_conv_centroid(conv, info["groups_global"])
            active_after = kernels_before
            unique_after = compact.num_bank_kernels
        else:
            keep_mask = torch.ones(kernels_before, dtype=torch.bool, device=conv.weight.device)
            if pruned_count > 0:
                keep_mask[torch.tensor(info["prune_idx"], device=conv.weight.device)] = False
            keep_idx = torch.where(keep_mask)[0]
            compact = CompactKernelConv2d.from_conv_representative(conv, keep_idx)
            active_after = int(keep_idx.numel())
            unique_after = active_after

        replace_module_by_name(model, name, compact.to(conv.weight.device))

        total_before += kernels_before
        total_removed_effective += merged_count
        total_active_after += active_after
        total_unique_after += unique_after
        layer_stats.append(
            {
                "layer_index": li,
                "layer_name": name,
                "option": option,
                "out_channels": info["out_channels"],
                "in_channels": info["in_channels"],
                "kernels_before": kernels_before,
                "active_kernels_after": active_after,
                "unique_kernels_after": unique_after,
                "num_candidates": info["num_candidates"],
                "num_groups": groups_count,
                "grouped_kernels": info["grouped_kernels"],
                "effective_removed_kernels": merged_count,
            }
        )

    prune_pct = 100.0 * total_removed_effective / max(total_before, 1)
    return {
        "kernel_option": option,
        "total_kernels_before": int(total_before),
        "total_removed_effective_kernels": int(total_removed_effective),
        "kernel_prune_pct": float(prune_pct),
        "total_active_kernels_after": int(total_active_after),
        "total_unique_kernels_after": int(total_unique_after),
        "kernel_layer_stats": layer_stats,
        "channel_scores": channel_scores,
        "kernel_prune_idx_map": kernel_prune_idx_map,
    }


def compute_kernel_prune_channel_scores(
    model: nn.Module,
    tau_s: float,
    norm_percent: float,
    max_candidates_per_layer: int,
    min_keep_channels: int,
    skip_stem: bool = True,
    skip_downsample: bool = True,
) -> Dict[str, torch.Tensor]:
    """
    Compute per-output-channel kernel-prune pressure WITHOUT mutating weights.

    score[name][c] = (# pruned kernels for output channel c) / Cin
    """
    out_prune_fraction: Dict[str, torch.Tensor] = {}
    conv_names = get_conv_layer_names(model)

    for name in conv_names:
        named = dict(model.named_modules())
        if name not in named or not isinstance(named[name], nn.Conv2d):
            continue
        if skip_stem and name in {"conv1", "features.conv0"}:
            continue
        if skip_downsample and "downsample" in name:
            continue

        conv = named[name]
        info = find_redundant_kernels(
            conv=conv,
            tau_s=tau_s,
            norm_percent=norm_percent,
            max_candidates=max_candidates_per_layer,
            min_keep=min_keep_channels,
            max_kernel_prune_frac_per_layer=1.0,
        )
        cin = conv.in_channels
        out_pruned = torch.zeros(conv.out_channels, device=conv.weight.device)
        if len(info["prune_idx"]) > 0:
            for idx in info["prune_idx"]:
                out_c = int(idx // cin)
                out_pruned[out_c] += 1.0
        out_prune_fraction[name] = out_pruned / max(cin, 1)
    return out_prune_fraction


def channel_prune_model(
    model: nn.Module,
    example_inputs: torch.Tensor,
    prune_ratio: float,
    min_keep_channels: int = 8,
    max_prune_channels_per_layer: int = 64,
    channel_scores: Optional[Dict[str, torch.Tensor]] = None,
    skip_stem: bool = True,
    skip_downsample: bool = True,
) -> Dict[str, object]:
    if tp is None:
        return {"total_pruned_out_channels": 0, "channel_layer_stats": [], "skipped": True, "reason": "torch_pruning_not_installed"}

    model.eval()
    total_pruned = 0
    stats: List[Dict[str, object]] = []
    conv_names = get_conv_layer_names(model)

    for li, name in enumerate(conv_names):
        named = dict(model.named_modules())
        if name not in named or not isinstance(named[name], nn.Conv2d):
            continue
        if skip_stem and name in {"conv1", "features.conv0"}:
            continue
        if skip_downsample and "downsample" in name:
            continue

        conv = named[name]
        out_before = int(conv.out_channels)
        max_prunable = max(0, out_before - min_keep_channels)
        if max_prunable <= 0:
            continue
        num_prune = min(int(out_before * prune_ratio), max_prunable, max_prune_channels_per_layer)
        if num_prune <= 0:
            continue

        if channel_scores is not None and name in channel_scores:
            # High pruned-kernel fraction => lower importance.
            score = channel_scores[name].detach().to(conv.weight.device)
            if score.numel() != out_before:
                importance = torch.norm(conv.weight.data.view(out_before, -1), p=2, dim=1)
            else:
                importance = 1.0 - score
        else:
            importance = torch.norm(conv.weight.data.view(out_before, -1), p=2, dim=1)

        prune_idx = torch.argsort(importance)[:num_prune].tolist()
        prune_idx = sorted(prune_idx, reverse=True)
        if len(prune_idx) == 0:
            continue

        # Must rebuild after each structural prune: the graph tracks module shapes/edges;
        # reusing one graph across iterations produces invalid groups (often ~chance accuracy).
        dg = tp.DependencyGraph().build_dependency(model, example_inputs=example_inputs)
        group = dg.get_pruning_group(conv, tp.prune_conv_out_channels, idxs=prune_idx)
        if not dg.check_pruning_group(group):
            continue
        group.prune()
        total_pruned += len(prune_idx)

        named2 = dict(model.named_modules())
        out_after = out_before
        if name in named2 and isinstance(named2[name], nn.Conv2d):
            out_after = int(named2[name].out_channels)
        stats.append(
            {
                "layer_index": li,
                "layer_name": name,
                "out_channels_before": out_before,
                "pruned_out_channels": len(prune_idx),
                "out_channels_after": out_after,
                "prune_ratio_target": prune_ratio,
            }
        )

    with torch.no_grad():
        probe = model(example_inputs)
        if not torch.isfinite(probe).all():
            raise RuntimeError(
                "Channel pruning produced non-finite activations on the tracing batch; "
                "aborting to avoid saving a broken checkpoint."
            )

    return {"total_pruned_out_channels": int(total_pruned), "channel_layer_stats": stats, "skipped": False, "reason": ""}


def fine_tune(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    device: torch.device,
    epochs: int = 20,
    lr: float = 1e-4,
    amp: bool = True,
) -> List[Dict[str, float]]:
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    use_amp = amp and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    hist: List[Dict[str, float]] = []
    for ep in range(epochs):
        model.train()
        total = 0
        run_loss = 0.0
        top1 = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total += y.size(0)
            run_loss += loss.item() * y.size(0)
            top1 += out.argmax(dim=1).eq(y).sum().item()

        scheduler.step()
        test_m = evaluate(model, test_loader, criterion, device)
        row = {
            "epoch": ep + 1,
            "train_loss": run_loss / max(total, 1),
            "train_top1": 100.0 * top1 / max(total, 1),
            "test_loss": test_m["loss"],
            "test_top1": test_m["top1"],
            "test_top5": test_m["top5"],
        }
        hist.append(row)
        print(
            f"Epoch {ep+1:02d}/{epochs} | Train Loss: {row['train_loss']:.4f} | "
            f"Train Top1: {row['train_top1']:.2f} | Test Top1: {row['test_top1']:.2f}",
            flush=True,
        )
    return hist


def load_phase1_model(
    checkpoint_path: str,
    device: torch.device,
    arch_override: Optional[str] = None,
    num_classes_override: Optional[int] = None,
) -> Tuple[nn.Module, Dict]:
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device)
    arch = (arch_override or ckpt.get("arch", "resnet34")).lower()
    num_classes = int(num_classes_override or ckpt.get("num_classes", 10))
    model = build_model(arch, num_classes).to(device)

    raw_state = ckpt["model_state_dict"]
    clean_state = {k: v for k, v in raw_state.items() if "total_ops" not in k and "total_params" not in k}
    model.load_state_dict(clean_state, strict=True)
    return model, ckpt


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Kernel-only pruning with real parameter reduction")
    parser.add_argument("--checkpoint", type=str, required=True, help="Path to baseline .pth")
    parser.add_argument("--data_root", type=str, default="./data")
    parser.add_argument("--dataset", type=str, default="cifar10", choices=["cifar10", "cifar100"])
    parser.add_argument("--output_dir", type=str, default="artifacts/phase2_3_4_kernel_only")
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--num_workers", type=int, default=4)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--tau_s", type=float, default=0.80)
    parser.add_argument("--norm_percent", type=float, default=60.0)
    parser.add_argument("--max_candidates_per_layer", type=int, default=1000)
    parser.add_argument("--min_keep_channels", type=int, default=8)
    parser.add_argument("--kernel_option", type=str, default="representative", choices=["representative", "centroid"])
    parser.add_argument(
        "--channel_prune_ratio",
        type=float,
        default=0.15,
        help="Non-grouped structural output-channel prune ratio after kernel pruning",
    )
    parser.add_argument(
        "--max_channel_prune_per_layer",
        type=int,
        default=64,
        help="Hard cap on pruned output channels per conv layer (prevents stacked destructive pruning across many layers)",
    )
    parser.add_argument(
        "--max_kernel_prune_frac_per_layer",
        type=float,
        default=0.12,
        help="Max fraction of kernels merged/pruned per conv when similarity groups degenerate (tau_s too low -> groups=1). "
        "1.0 disables this safety cap.",
    )
    parser.add_argument(
        "--include_stem",
        action="store_true",
        help="If set, include stem conv (conv1/features.conv0) in kernel pruning",
    )
    parser.add_argument(
        "--include_downsample",
        action="store_true",
        help="If set, include residual downsample conv layers in kernel pruning",
    )
    parser.add_argument("--amp", action="store_true")
    parser.add_argument("--skip_flops", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args, _ = parser.parse_known_args()
    return args


def main() -> None:
    args = parse_args()
    set_seed(args.seed)
    device = torch.device(args.device)
    pin_memory = device.type == "cuda"

    os.makedirs(args.output_dir, exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(args.output_dir, "tables"), exist_ok=True)

    baseline_model, ckpt = load_phase1_model(args.checkpoint, device)
    arch = ckpt.get("arch", "unknown").lower()
    if arch not in {"resnet18", "resnet34", "densenet121"}:
        raise ValueError(f"Unsupported arch from checkpoint: {arch}")

    train_loader, test_loader, _ = get_cifar_loaders(
        data_root=args.data_root,
        dataset_name=args.dataset,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        pin_memory=pin_memory,
    )
    criterion = nn.CrossEntropyLoss()

    baseline_metrics = evaluate(baseline_model, test_loader, criterion, device)
    baseline_params = count_params(baseline_model)
    baseline_flops = float("nan") if args.skip_flops else compute_flops(baseline_model, device)

    baseline_ckpt_path = os.path.join(args.output_dir, "checkpoints", f"{arch}_{args.dataset}_baseline_snapshot.pth")
    torch.save(
        {
            "arch": arch,
            "dataset": args.dataset,
            "stage": "baseline",
            "model_state_dict": baseline_model.state_dict(),
            "metrics": baseline_metrics,
            "params": baseline_params,
            "flops": baseline_flops,
        },
        baseline_ckpt_path,
    )

    print("\n=== Kernel Pruning (Only) ===", flush=True)
    print(
        f"[Config] tau_s={args.tau_s} norm_percent={args.norm_percent} max_candidates_per_layer={args.max_candidates_per_layer} "
        f"max_kernel_prune_frac_per_layer={args.max_kernel_prune_frac_per_layer} kernel_option={args.kernel_option}",
        flush=True,
    )
    pruned_model = copy.deepcopy(baseline_model).to(device)
    prune_info = kernel_prune_model(
        model=pruned_model,
        tau_s=args.tau_s,
        norm_percent=args.norm_percent,
        max_candidates_per_layer=args.max_candidates_per_layer,
        min_keep_channels=args.min_keep_channels,
        kernel_option=args.kernel_option,
        max_kernel_prune_frac_per_layer=float(args.max_kernel_prune_frac_per_layer),
        skip_stem=(not args.include_stem),
        skip_downsample=(not args.include_downsample),
    )
    post_prune_metrics = evaluate(pruned_model, test_loader, criterion, device)
    post_prune_params = count_params(pruned_model)
    post_prune_flops = float("nan") if args.skip_flops else compute_flops(pruned_model, device)
    param_reduction_pct = 100.0 * (baseline_params - post_prune_params) / max(baseline_params, 1)
    flops_reduction_pct = (
        np.nan
        if (args.skip_flops or np.isnan(baseline_flops) or np.isnan(post_prune_flops) or baseline_flops == 0)
        else 100.0 * (baseline_flops - post_prune_flops) / baseline_flops
    )

    print("\n=== Baseline vs Post-Kernel-Prune (Before FT) ===", flush=True)
    print(
        f"Baseline   -> Loss: {baseline_metrics['loss']:.6f}, Top1: {baseline_metrics['top1']:.4f}, Top5: {baseline_metrics['top5']:.4f}",
        flush=True,
    )
    print(
        f"PostKernel -> Loss: {post_prune_metrics['loss']:.6f}, Top1: {post_prune_metrics['top1']:.4f}, Top5: {post_prune_metrics['top5']:.4f}",
        flush=True,
    )
    print(
        f"Total kernels before pruning: {prune_info['total_kernels_before']} | "
        f"Effective removed kernels: {prune_info['total_removed_effective_kernels']}",
        flush=True,
    )
    print(
        f"Kernel prune %: {prune_info['kernel_prune_pct']:.4f} | "
        f"Param reduction %: {param_reduction_pct:.4f} | "
        f"FLOPs reduction %: {flops_reduction_pct if not np.isnan(flops_reduction_pct) else float('nan'):.4f}",
        flush=True,
    )
    print(
        f"Active kernels after: {prune_info['total_active_kernels_after']} | "
        f"Unique kernels after: {prune_info['total_unique_kernels_after']}",
        flush=True,
    )

    # Stage B: non-grouped channel pruning on the post-kernel weight state.
    # Since Stage A uses CompactKernelConv2d, we replay the exact kernel removals as zero-masks
    # on a dense Conv2d clone and run dependency-aware channel pruning on that model.
    print("\n=== Non-Grouped Channel Pruning (After Kernel Pruning) ===", flush=True)
    print(
        f"[Config] channel_prune_ratio={float(args.channel_prune_ratio):.4f} "
        f"max_channel_prune_per_layer={int(args.max_channel_prune_per_layer)} "
        f"min_keep_channels={int(args.min_keep_channels)}",
        flush=True,
    )
    channel_model = copy.deepcopy(baseline_model).to(device)
    apply_kernel_prune_mask_from_map(channel_model, prune_info.get("kernel_prune_idx_map", {}))
    post_kernel_masked_params = count_params(channel_model)
    score_map = prune_info.get("channel_scores", {})
    example_inputs = torch.randn(1, 3, 224, 224, device=device)
    channel_info = channel_prune_model(
        model=channel_model,
        example_inputs=example_inputs,
        prune_ratio=args.channel_prune_ratio,
        min_keep_channels=args.min_keep_channels,
        max_prune_channels_per_layer=int(args.max_channel_prune_per_layer),
        channel_scores=score_map,
        skip_stem=(not args.include_stem),
        skip_downsample=(not args.include_downsample),
    )
    channel_metrics = evaluate(channel_model, test_loader, criterion, device)
    channel_params = count_params(channel_model)
    channel_flops = float("nan") if args.skip_flops else compute_flops(channel_model, device)
    channel_param_reduction_pct = 100.0 * (baseline_params - channel_params) / max(baseline_params, 1)
    channel_flops_reduction_pct = (
        np.nan
        if (args.skip_flops or np.isnan(baseline_flops) or np.isnan(channel_flops) or baseline_flops == 0)
        else 100.0 * (baseline_flops - channel_flops) / baseline_flops
    )
    print(
        f"PostChannel -> Loss: {channel_metrics['loss']:.6f}, Top1: {channel_metrics['top1']:.4f}, Top5: {channel_metrics['top5']:.4f}",
        flush=True,
    )
    print(
        f"Pruned out-channels: {channel_info['total_pruned_out_channels']} | "
        f"Param reduction %: {channel_param_reduction_pct:.4f} | "
        f"FLOPs reduction %: {channel_flops_reduction_pct if not np.isnan(channel_flops_reduction_pct) else float('nan'):.4f}",
        flush=True,
    )
    kernel_to_channel_param_drop_pct = (
        100.0 * (post_kernel_masked_params - channel_params) / max(post_kernel_masked_params, 1)
    )
    print(
        f"Params: baseline={baseline_params} | post_kernel_compact={post_prune_params} | "
        f"post_kernel_masked={post_kernel_masked_params} | post_channel={channel_params} | "
        f"extra_drop_vs_kernel_input_params_%={kernel_to_channel_param_drop_pct:.4f}",
        flush=True,
    )
    print(
        f"Channel scoring: used_kernel_scores_layers={len(score_map)} | "
        f"max_channel_prune_per_layer={int(args.max_channel_prune_per_layer)} | "
        f"channel_prune_ratio={float(args.channel_prune_ratio):.4f}",
        flush=True,
    )
    if channel_info["skipped"]:
        print("Warning: channel pruning skipped because torch_pruning is unavailable.", flush=True)

    post_prune_ckpt_path = os.path.join(args.output_dir, "checkpoints", f"{arch}_{args.dataset}_post_kernel_pre_ft.pth")
    torch.save(
        {
            "arch": arch,
            "dataset": args.dataset,
            "stage": "post_kernel_pre_ft",
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "kernel_option": args.kernel_option,
            "model_state_dict": pruned_model.state_dict(),
            "metrics": post_prune_metrics,
            "prune_info": prune_info,
            "params": post_prune_params,
            "flops": post_prune_flops,
        },
        post_prune_ckpt_path,
    )
    post_channel_ckpt_path = os.path.join(
        args.output_dir, "checkpoints", f"{arch}_{args.dataset}_post_kernel_channel_pre_ft.pth"
    )
    torch.save(
        {
            "arch": arch,
            "dataset": args.dataset,
            "stage": "post_kernel_channel_pre_ft",
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "channel_prune_ratio": args.channel_prune_ratio,
            "kernel_option": args.kernel_option,
            "model_state_dict": channel_model.state_dict(),
            "metrics": channel_metrics,
            "prune_info": prune_info,
            "channel_prune_info": channel_info,
            "params": channel_params,
            "flops": channel_flops,
        },
        post_channel_ckpt_path,
    )

    print("\n=== Fine-tuning (full) ===", flush=True)
    ft_model = copy.deepcopy(channel_model).to(device)
    ft_history = fine_tune(
        model=ft_model,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=args.epochs,
        lr=args.lr,
        amp=args.amp,
    )
    post_ft_metrics = evaluate(ft_model, test_loader, criterion, device)
    post_ft_params = count_params(ft_model)
    post_ft_flops = float("nan") if args.skip_flops else compute_flops(ft_model, device)

    post_ft_ckpt_path = os.path.join(args.output_dir, "checkpoints", f"{arch}_{args.dataset}_post_kernel_post_ft_full.pth")
    torch.save(
        {
            "arch": arch,
            "dataset": args.dataset,
            "stage": "post_kernel_post_ft_full",
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "kernel_option": args.kernel_option,
            "model_state_dict": ft_model.state_dict(),
            "metrics": post_ft_metrics,
            "prune_info": prune_info,
            "finetune_history": ft_history,
            "params": post_ft_params,
            "flops": post_ft_flops,
        },
        post_ft_ckpt_path,
    )

    layer_stats_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_layer_stats_kernel.csv")
    pd.DataFrame(prune_info["kernel_layer_stats"]).to_csv(layer_stats_path, index=False)
    channel_stats_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_layer_stats_channel.csv")
    pd.DataFrame(channel_info["channel_layer_stats"]).to_csv(channel_stats_path, index=False)
    history_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_finetune_history_full.csv")
    pd.DataFrame(ft_history).to_csv(history_path, index=False)

    metrics_rows = [
        {
            "stage": "baseline",
            "top1": baseline_metrics["top1"],
            "top5": baseline_metrics["top5"],
            "loss": baseline_metrics["loss"],
            "params": baseline_params,
            "flops": baseline_flops,
            "tau_s": np.nan,
            "norm_percent": np.nan,
            "kernel_option": args.kernel_option,
            "total_kernels_before": prune_info["total_kernels_before"],
            "effective_removed_kernels": 0,
            "kernel_prune_%": 0.0,
            "pruned_out_channels_total": 0,
            "param_reduction_%": 0.0,
            "flops_reduction_%": 0.0 if not np.isnan(baseline_flops) else np.nan,
            "checkpoint_path": baseline_ckpt_path,
        },
        {
            "stage": "post_kernel_pre_ft",
            "top1": post_prune_metrics["top1"],
            "top5": post_prune_metrics["top5"],
            "loss": post_prune_metrics["loss"],
            "params": post_prune_params,
            "flops": post_prune_flops,
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "kernel_option": args.kernel_option,
            "total_kernels_before": prune_info["total_kernels_before"],
            "effective_removed_kernels": prune_info["total_removed_effective_kernels"],
            "kernel_prune_%": prune_info["kernel_prune_pct"],
            "pruned_out_channels_total": 0,
            "param_reduction_%": param_reduction_pct,
            "flops_reduction_%": flops_reduction_pct,
            "checkpoint_path": post_prune_ckpt_path,
        },
        {
            "stage": "post_kernel_channel_pre_ft",
            "top1": channel_metrics["top1"],
            "top5": channel_metrics["top5"],
            "loss": channel_metrics["loss"],
            "params": channel_params,
            "flops": channel_flops,
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "kernel_option": args.kernel_option,
            "total_kernels_before": prune_info["total_kernels_before"],
            "effective_removed_kernels": prune_info["total_removed_effective_kernels"],
            "kernel_prune_%": prune_info["kernel_prune_pct"],
            "pruned_out_channels_total": channel_info["total_pruned_out_channels"],
            "param_reduction_%": channel_param_reduction_pct,
            "flops_reduction_%": channel_flops_reduction_pct,
            "checkpoint_path": post_channel_ckpt_path,
        },
        {
            "stage": "post_kernel_post_ft_full",
            "top1": post_ft_metrics["top1"],
            "top5": post_ft_metrics["top5"],
            "loss": post_ft_metrics["loss"],
            "params": post_ft_params,
            "flops": post_ft_flops,
            "tau_s": args.tau_s,
            "norm_percent": args.norm_percent,
            "kernel_option": args.kernel_option,
            "total_kernels_before": prune_info["total_kernels_before"],
            "effective_removed_kernels": prune_info["total_removed_effective_kernels"],
            "kernel_prune_%": prune_info["kernel_prune_pct"],
            "pruned_out_channels_total": channel_info["total_pruned_out_channels"],
            "param_reduction_%": 100.0 * (baseline_params - post_ft_params) / max(baseline_params, 1),
            "flops_reduction_%": (
                np.nan
                if (args.skip_flops or np.isnan(baseline_flops) or np.isnan(post_ft_flops) or baseline_flops == 0)
                else 100.0 * (baseline_flops - post_ft_flops) / baseline_flops
            ),
            "checkpoint_path": post_ft_ckpt_path,
        },
    ]

    for row in metrics_rows:
        row["acc_drop_after_prune"] = baseline_metrics["top1"] - channel_metrics["top1"]
        row["loss_increase_after_prune"] = channel_metrics["loss"] - baseline_metrics["loss"]
        if row["stage"] == "post_kernel_post_ft_full":
            row["acc_recovery_after_ft"] = row["top1"] - channel_metrics["top1"]
            row["loss_recovery_after_ft"] = channel_metrics["loss"] - row["loss"]
        else:
            row["acc_recovery_after_ft"] = np.nan
            row["loss_recovery_after_ft"] = np.nan

    metrics_path = os.path.join(args.output_dir, "tables", f"{arch}_{args.dataset}_metrics_table.csv")
    pd.DataFrame(metrics_rows).to_csv(metrics_path, index=False)

    print("\nDone. Saved artifacts:", flush=True)
    print(f"- {baseline_ckpt_path}", flush=True)
    print(f"- {post_prune_ckpt_path}", flush=True)
    print(f"- {post_channel_ckpt_path}", flush=True)
    print(f"- {post_ft_ckpt_path}", flush=True)
    print(f"- {layer_stats_path}", flush=True)
    print(f"- {channel_stats_path}", flush=True)
    print(f"- {history_path}", flush=True)
    print(f"- {metrics_path}", flush=True)



In [8]:
import sys

# Save the original sys.argv
original_argv = sys.argv

# Simulate command-line arguments to provide the required --checkpoint
sys.argv = ['main.py', '--checkpoint', 'resnet34_cifar10_baseline_final.pth']

try:
    main()
finally:
    # Restore sys.argv to its original state to avoid interfering with other cells
    sys.argv = original_argv

Files already downloaded and verified
Files already downloaded and verified

=== Kernel Pruning (Only) ===
[Config] tau_s=0.8 norm_percent=60.0 max_candidates_per_layer=1000 max_kernel_prune_frac_per_layer=0.12 kernel_option=representative
[Kernel Pruning] Layer 2/36: layer1.0.conv1
Layer: layer1.0.conv1 | candidates=1000 | groups=0 | effective_removed=491
[Kernel Pruning] Layer 3/36: layer1.0.conv2
Layer: layer1.0.conv2 | candidates=1000 | groups=0 | effective_removed=491
[Kernel Pruning] Layer 4/36: layer1.1.conv1
Layer: layer1.1.conv1 | candidates=1000 | groups=0 | effective_removed=491
[Kernel Pruning] Layer 5/36: layer1.1.conv2
Layer: layer1.1.conv2 | candidates=1000 | groups=0 | effective_removed=491
[Kernel Pruning] Layer 6/36: layer1.2.conv1
Layer: layer1.2.conv1 | candidates=1000 | groups=0 | effective_removed=491
[Kernel Pruning] Layer 7/36: layer1.2.conv2
Layer: layer1.2.conv2 | candidates=1000 | groups=0 | effective_removed=491
[Kernel Pruning] Layer 8/36: layer2.0.conv1
La

In [2]:
import torch
torch.cuda.set_device(4)